# SAM 3 on SageMaker Async Inference — text-prompt masking

Deploy Meta's [SAM 3](https://huggingface.co/facebook/sam3) (Segment Anything Model 3) behind a SageMaker **async** endpoint and segment objects by **text prompt**.

SAM 3 does **Promptable Concept Segmentation (PCS)**: give it an image and a short noun phrase like `"chair"` and it returns instance masks for *every* chair in the picture — no clicks, no boxes. That is exactly the "type a word, mask the thing" behaviour we want.

**Why `facebook/sam3` and not `facebook/sam3.1`?** SAM 3.1 is the newest checkpoint but ships only as a raw `.pt` (`library_name: checkpoint`, no `transformers` integration) and needs Meta's `facebookresearch/sam3` package to load. `facebook/sam3` has native `transformers` support (`Sam3Model` / `Sam3Processor`, `model.safetensors`), so the handler is a clean `from_pretrained` on a stock PyTorch DLC. The PCS text-prompt capability is identical on a single image; 3.1's headline gain (Object Multiplex) is about faster multi-object *video* tracking. Swapping up later is a weights-prefix change.

**Layout** (mirrors the hardened `FLUX2-Klein-sagemaker` pattern in this workspace):
- Weights live as uncompressed objects under an S3 prefix, mounted verbatim to `/opt/ml/model/`.
- `code/inference.py` + `code/requirements.txt` live in a `code/` subfolder *inside* that prefix — the toolkit auto-discovers `/opt/ml/model/code/inference.py`. Editing the handler = re-upload a few KB, never re-archive weights.
- The container never talks to the Hub at runtime and holds no HF token.

In [ ]:
%pip install -q "sagemaker>=2.254.1" boto3

## 1. Setup

In [ ]:
import boto3
import sagemaker

sess = sagemaker.Session()
region = sess.boto_region_name
bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except Exception:
    # Running outside SageMaker (e.g. laptop): set a role ARN with SageMaker + S3 access.
    role = boto3.client("iam").get_role(RoleName="<your-sagemaker-execution-role>")["Role"]["Arn"]

# --- Configuration -------------------------------------------------------
MODEL_ID        = "facebook/sam3"
WEIGHTS_PREFIX  = "sam3/weights"                 # S3 prefix that holds weights + code/
ENDPOINT_NAME   = "sam3-pcs"

# Best cost/performance default. See the README cost section for the trade-offs:
#   ml.g6.xlarge   -> NVIDIA L4  (24 GB, Ada)   best $/perf, recommended
#   ml.g5.xlarge   -> NVIDIA A10G (24 GB, Ampere) proven fallback (used elsewhere in this repo)
#   ml.g4dn.xlarge -> NVIDIA T4  (16 GB, Turing) cheapest; still fits SAM 3, just slower
INSTANCE_TYPE   = "ml.g6.xlarge"
# PyTorch 2.6 inference DLC (py312, CUDA 12.4). 2.6 is the newest image URI the
# sagemaker<3.0 SDK knows about; on torch 2.6 we must keep transformers <5.0
# (5.x imports torch.float8_e8m0fnu, which only exists on torch>=2.7).
FRAMEWORK_VER   = "2.6"
PY_VER          = "py312"

weights_s3_uri  = f"s3://{bucket}/{WEIGHTS_PREFIX}"
print(f"region={region}\nbucket={bucket}\nweights={weights_s3_uri}\nrole={role}")

## 2. Stage weights to S3 (one-time)

`facebook/sam3` is **gated** — accept the license at https://huggingface.co/facebook/sam3 and use an access-granted `HF_TOKEN`.

Run this once from anywhere with `HF_TOKEN` exported and the AWS CLI configured (this notebook instance is fine). Skip if the prefix is already populated.

```bash
export HF_TOKEN=hf_...
pip install "huggingface_hub>=0.26" hf_transfer
HF_HUB_ENABLE_HF_TRANSFER=1 python prepare_weights.py --out weights
aws s3 sync weights/ s3://<bucket>/sam3/weights/
```

The cell below runs the same thing from the notebook. It needs `HF_TOKEN` in the environment.

In [ ]:
# Optional: stage weights from the notebook. Comment out if you ran the CLI steps above.
import os, subprocess, sys

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN (with facebook/sam3 access granted) before running this cell."
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.26", "hf_transfer"])

env = {**os.environ, "HF_HUB_ENABLE_HF_TRANSFER": "1"}
subprocess.check_call([sys.executable, "prepare_weights.py", "--out", "weights"], env=env)
subprocess.check_call(["aws", "s3", "sync", "weights/", f"{weights_s3_uri}/"])
print("Weights synced to", weights_s3_uri)

## 3. Upload the `code/` bundle into the weights prefix

With uncompressed-prefix `model_data` we do **not** pass `entry_point` / `source_dir` (that triggers the SDK repack path, which silently drops a dict `model_data`). Instead the handler lives at `<prefix>/code/inference.py` and the toolkit picks it up automatically. Re-run this cell after any edit to `inference.py`.

In [ ]:
s3 = boto3.client("s3")
for fname in ["inference.py", "requirements.txt"]:
    s3.upload_file(f"code/{fname}", bucket, f"{WEIGHTS_PREFIX}/code/{fname}")
    print(f"uploaded code/{fname} -> s3://{bucket}/{WEIGHTS_PREFIX}/code/{fname}")

## 4. Deploy the async endpoint

First deploy takes ~5-8 min (image pull + `pip install transformers` + ~3.4 GB weight mount). Async because we read the input image from S3 and write JSON results back to S3, sidestepping the real-time endpoint's 6 MB / 60 s limits.

In [ ]:
from sagemaker.pytorch import PyTorchModel
from sagemaker.async_inference import AsyncInferenceConfig

async_config = AsyncInferenceConfig(
    output_path=f"s3://{bucket}/sam3-outputs/",
    failure_path=f"s3://{bucket}/sam3-failures/",
    max_concurrent_invocations_per_instance=2,
)

model_data = {
    "S3DataSource": {
        "S3Uri": f"{weights_s3_uri}/",
        "S3DataType": "S3Prefix",
        "CompressionType": "None",
    }
}

model = PyTorchModel(
    role=role,
    model_data=model_data,
    framework_version=FRAMEWORK_VER,
    py_version=PY_VER,
    env={
        "SAGEMAKER_MODEL_SERVER_TIMEOUT": "3600",
        "TS_DEFAULT_RESPONSE_TIMEOUT": "3600",
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        "SAM3_DTYPE": "float32",   # set bfloat16 / float16 to trade accuracy for speed
    },
)

# Clean slate so we never redeploy on top of a stale model/config.
sm = boto3.client("sagemaker")
for _name in [ENDPOINT_NAME]:
    for _del in (lambda: sm.delete_endpoint(EndpointName=_name),
                 lambda: sm.delete_endpoint_config(EndpointConfigName=_name)):
        try:
            _del()
        except sm.exceptions.ClientError:
            pass

predictor = model.deploy(
    endpoint_name=ENDPOINT_NAME,
    endpoint_config_name=ENDPOINT_NAME,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    async_inference_config=async_config,
    container_startup_health_check_timeout=1800,
)
print("Deployed:", ENDPOINT_NAME)

## 5. Invocation helper

Async flow: PUT the request JSON (base64 image + text prompt) to S3, call `invoke_endpoint_async`, poll the returned `OutputLocation` until the result lands.

In [ ]:
import base64, json, time, uuid
from urllib.parse import urlparse

sm_runtime = boto3.client("sagemaker-runtime")
s3 = boto3.client("s3")

def _s3_get(uri):
    p = urlparse(uri)
    return s3.get_object(Bucket=p.netloc, Key=p.path.lstrip("/"))["Body"].read()

def _s3_wait(uri, timeout=600, poll=3):
    p = urlparse(uri); key = p.path.lstrip("/")
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            s3.head_object(Bucket=p.netloc, Key=key)
            return True
        except s3.exceptions.ClientError:
            time.sleep(poll)
    return False

def segment(image_path, text, threshold=0.5, mask_threshold=0.5, max_instances=100):
    """Run SAM 3 PCS on a local image with a text concept prompt. Returns the parsed result dict."""
    with open(image_path, "rb") as f:
        image_b64 = base64.b64encode(f.read()).decode("ascii")
    payload = {
        "image": image_b64,
        "text": text,
        "threshold": threshold,
        "mask_threshold": mask_threshold,
        "max_instances": max_instances,
    }
    in_key = f"sam3-inputs/req-{uuid.uuid4().hex}.json"
    s3.put_object(Bucket=bucket, Key=in_key, Body=json.dumps(payload).encode("utf-8"))

    resp = sm_runtime.invoke_endpoint_async(
        EndpointName=ENDPOINT_NAME,
        InputLocation=f"s3://{bucket}/{in_key}",
        ContentType="application/json",
    )
    out_uri = resp["OutputLocation"]
    fail_uri = resp.get("FailureLocation")
    print("submitted; waiting on", out_uri)

    while True:
        if _s3_wait(out_uri, timeout=5, poll=2):
            return json.loads(_s3_get(out_uri))
        if fail_uri and _s3_wait(fail_uri, timeout=1, poll=1):
            raise RuntimeError(_s3_get(fail_uri).decode("utf-8", "replace"))

## 6. Test: mask a concept with a text prompt and mark it on the original

Assumes `sample.png` sits next to this notebook. Change `PROMPT` to whatever you want to segment (e.g. `"chair"`, `"person"`, `"dog"`).

In [ ]:
PROMPT = "chair"
result = segment("sample.png", PROMPT, threshold=0.5)
print(f"prompt={result['prompt']!r}  instances={result['num_instances']}  image_size={result['image_size']}")
print("top scores:", [round(s, 3) for s in result['scores'][:10]])

In [ ]:
import base64, io, colorsys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def _distinct_colors(n):
    return [tuple(int(255 * c) for c in colorsys.hsv_to_rgb(i / max(n, 1), 0.65, 1.0)) for i in range(n)]

def mark_masks(image_path, result, alpha=0.5):
    """Overlay every returned instance mask onto the original image (masks only, no boxes)."""
    base = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    colors = _distinct_colors(len(result["masks"]))

    for color, mask_b64 in zip(colors, result["masks"]):
        m = np.array(Image.open(io.BytesIO(base64.b64decode(mask_b64))).convert("L")) > 127
        rgba = np.zeros((*m.shape, 4), dtype=np.uint8)
        rgba[m] = (*color, int(255 * alpha))
        overlay = Image.alpha_composite(overlay, Image.fromarray(rgba, "RGBA"))

    marked = Image.alpha_composite(base, overlay).convert("RGB")
    return marked

marked = mark_masks("sample.png", result)
marked.save("marked_output.png")
print("saved marked_output.png")

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
ax[0].imshow(Image.open("sample.png")); ax[0].set_title("original"); ax[0].axis("off")
ax[1].imshow(marked); ax[1].set_title(f"SAM 3: \"{result['prompt']}\" ({result['num_instances']} found)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 7. Cleanup

The GPU endpoint is the main cost — delete it when idle. The `sam3/weights/` prefix can stay for fast redeploys; the `sam3-inputs/`, `sam3-outputs/`, `sam3-failures/` prefixes are good candidates for an S3 lifecycle rule.

In [ ]:
predictor.delete_endpoint()
try:
    sess.delete_model(model.name)
except Exception as e:
    print(e)
print("torn down")